In [ ]:
import os
import gc
import traceback

import cv2
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, hsv_to_rgb
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.patches as patches
from numba import jit
from PIL import Image
from skimage import morphology, measure

import cfospy

In [ ]:
def mask_range(mask0):
    """Return a bounding box [xmin, xmax, ymin, ymax, zmin, zmax] for a given mask array."""
    v_ind = np.where(mask0)

    xmin = np.min(v_ind[2])
    ymin = np.min(v_ind[1])
    zmin = np.min(v_ind[0])
    xmax = np.max(v_ind[2])
    ymax = np.max(v_ind[1])
    zmax = np.max(v_ind[0])

    if xmin < 0:
        xmin = 0
    if xmax > ca.x_num:
        xmax = ca.x_num
    if ymin < 0:
        ymin = 0
    if ymax > ca.y_num:
        ymax = ca.y_num
    if zmin < 0:
        zmin = 0
    if zmax > ca.z_num:
        zmax = ca.z_num

    return xmin, xmax, ymin, ymax, zmin, zmax


def mask_range_padded(IDs, r):
    """Return a padded bounding box [xmin, xmax, ymin, ymax, zmin, zmax] for given IDs."""
    ID_li = []
    for rID in IDs:
        if not ca.smallID_q(rID):
            child_IDs, child_regions, middle_IDs, middle_regions = ca.get_child_IDs2(rID)
            ID_li += child_IDs, middle_IDs
        else:
            ID_li += [rID]

    mask0 = np.isin(np.swapaxes((ca.voxel_ID_order_all).reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2), ID_li[0])
    v_ind = np.where(mask0)

    xmin = np.min(v_ind[2])
    ymin = np.min(v_ind[1])
    zmin = np.min(v_ind[0])
    xmax = np.max(v_ind[2])
    ymax = np.max(v_ind[1])
    zmax = np.max(v_ind[0])

    xmin = int(xmin - (xmax - xmin) / r)
    ymin = int(ymin - (ymax - ymin) / r)
    zmin = int(zmin - (zmax - zmin) / r)
    xmax = int(xmax + (xmax - xmin) / r)
    ymax = int(ymax + (ymax - ymin) / r)
    zmax = int(zmax + (zmax - zmin) / r)

    if xmin < 0:
        xmin = 0
    if xmax > ca.x_num:
        xmax = ca.x_num
    if ymin < 0:
        ymin = 0
    if ymax > ca.y_num:
        ymax = ca.y_num
    if zmin < 0:
        zmin = 0
    if zmax > ca.z_num:
        zmax = ca.z_num

    return xmin, xmax, ymin, ymax, zmin, zmax


def mask_range_hemi(mask0):
    """Return hemisphere-wise bounding boxes for a given mask array."""
    v_ind = np.where(mask0)

    xmin = np.min(v_ind[2])
    ymin = np.min(v_ind[1])
    zmin = np.min(v_ind[0])
    xmax = np.max(v_ind[2])
    ymax = np.max(v_ind[1])
    zmax = np.max(v_ind[0])

    if xmin < 0:
        xmin = 0
    if xmax > ca.x_num:
        xmax = ca.x_num
    if ymin < 0:
        ymin = 0
    if ymax > ca.y_num:
        ymax = ca.y_num
    if zmin < 0:
        zmin = 0
    if zmax > ca.z_num:
        zmax = ca.z_num

    xmin_l, xmax_l = xmin, xmax
    xmin_r, xmax_r = xmin, xmax

    return xmin_l, xmax_l, xmin_r, xmax_r, ymin, ymax, zmin, zmax


@jit(nopython=True)
def make_edge(rID, mask):
    """Return edge voxel coordinates for a given 3D mask."""
    nonzero_indices = np.nonzero(mask)

    list = []
    for z, y, x in zip(*nonzero_indices):
        if z < 1 or z > mask.shape[0] - 2:
            continue
        if y < 1 or y > mask.shape[1] - 2:
            continue
        if x < 1 or x > mask.shape[2] - 2:
            continue

        is_edge = False
        for i in range(-1, 2):
            for j in range(-1, 2):
                for k in range(-1, 2):
                    if abs(i) + abs(j) + abs(k) == 1:
                        if mask[z + i, y + j, x + k] == 0:
                            is_edge = True
                            break
                if is_edge:
                    break
            if is_edge:
                break
        if is_edge:
            list.append((z, y, x, rID))
    return list


def trim_image(input_path, output_path, margin_top=0):
    """Trim outer margins from a PNG image and save it."""
    # Read PNG image
    image = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)
    print(image.shape)

    # Create binary mask for non-white pixels
    image_zero = np.zeros((image.shape[0], image.shape[1]), dtype="uint8")
    image_zero[np.where(image != (255, 255, 255, 255))[0:2]] = 255

    # Get bounding box of non-white region
    coords = cv2.findNonZero(image_zero)
    x, y, w, h = cv2.boundingRect(coords)

    # Shift y upward by margin_top and extend height accordingly
    y = max(y - margin_top, 0)
    h = min(h + margin_top, image.shape[0] - y)
    trimmed_image = image[y:y + h, x:x + w]

    # Display trimmed image
    plt.figure(figsize=(10, 10))
    plt.imshow(trimmed_image)
    plt.axis("off")
    plt.show()

    # Save output image
    cv2.imwrite(output_path, trimmed_image)

    
def trim_image_pil(input_path, output_path):
    """Trim outer margins from a PNG image and save it using PIL."""
    # Read PNG image
    image = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

    # Create binary mask for non-white pixels
    image_zero = np.zeros((image.shape[0], image.shape[1]), dtype="uint8")
    image_zero[np.where(image != (255, 255, 255, 255))[0:2]] = 255

    # Get bounding box of non-white region
    coords = cv2.findNonZero(image_zero)
    x, y, w, h = cv2.boundingRect(coords)

    # Crop the image
    trimmed_image = image[y:y + h, x:x + w]

    # Convert to RGB and save with PIL
    trimmed_image = Image.fromarray(cv2.cvtColor(trimmed_image, cv2.COLOR_BGR2RGB))
    dpi = (600, 600)
    trimmed_image.save(output_path, dpi=dpi)


def thicken_specific_lines(image, coordinates, radius=2):
    """Draw thick regional edges."""
    # Create binary mask from given coordinates
    mask = np.zeros_like(image, dtype=bool)
    for z, y, x, _ in coordinates:
        mask[z, y, x] = True

    # Define structuring element
    selem = morphology.ball(radius)

    # Dilate mask
    dilated_mask = morphology.binary_dilation(mask, selem)
    dilated_coords = np.where(dilated_mask == 1)

    # Combine with original image
    result = np.logical_or(image.astype(bool), dilated_mask)

    return result.astype(np.uint8), dilated_coords

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"
savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Load atlas data

vx = 20
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")

# Read atlas data
ca = cfospy.analysis.read_atlas_data(rdir, vx)
atlas_mask = ca.get_atlas_mask()
print(f"{len(ca.ID_all)} regions")

# Get unique region IDs and their reverse mapping
uni_IDs, rev_IDs = ca.get_uni_rIDs()

# Get region summary
df_sum = ca.get_sum_temp(uni_IDs)

# Convert RGB string to normalized triplets
df_sum["rgb_triplet2"] = df_sum["rgb_triplet"].apply(lambda x: np.array(list(map(int, x.strip("[]").split(", ")))) / 255)
print(df_sum)

In [ ]:
# IDs of medial brain regions
medial_regions = [830, 347, 4, 302, 31, 935, 48, 588, 972, 171, 44, 707, 714, 731, 484, 589, 597, 605, 19, 564, 609, 59, 571, 181, 56559, 189, 599, 30, 118, 223, 272, 763, 126, 133, 338, 491, 732, 60655, 60659, 525, 63, 10671, 59923, 323, 795, 634, 165, 12, 100, 60838, 60834, 60842, 591, 898, 679, 604, 354, 1048, 154, 169, 773, 949, 336, 117, 62, 158, 744, 198, 397, 530, 449, 611, 140]

In [ ]:
# Read rhythmicity data

cos_dir = os.path.join(src, "cos_results")
res = "cos.cell_count_1st2nd_ai_fpr0.5.csv"

# Read cosinor test results
ct_path = os.path.join(cos_dir, res)
CT_df = pd.read_csv(ct_path)

CT_df

In [ ]:
# Make 3D gray whole-brain image with regional masks

fig_dir = os.path.join(savedir, "whole_3D_phase")
os.makedirs(fig_dir, exist_ok=True)

r = 5
angles = ["hor", "cor", "sag"]
region_IDs = [147, 909, 726, 830, 162, 1061, 302, 382, 4, 385, 1002, 347, 1079]

brain_IDs = [315, 698, 1089, 703, 477, 803, 549, 1097, 313, 771, 354, 512]

xmin, xmax, ymin, ymax, zmin, zmax = mask_range_padded(brain_IDs, r)

y_scale = 1.5

brain_masks = []
face_colors = []
for rID in brain_IDs:
    ind = ca.get_vx_ind(rID)
    brain_mask = np.zeros(ca.voxel_nums, dtype="uint16")
    brain_mask[ind] = 1
    brain_mask = np.swapaxes(brain_mask.reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2)
    brain_mask = np.repeat(brain_mask, y_scale, axis=1)

    brain_masks.append(brain_mask)
    if rID == 315:
        face_colors.append((0.78, 0.78, 0.78))
    else:
        face_colors.append((0.93, 0.93, 0.93))

for region_ID in region_IDs:
    region = ca.df_allen[ca.df_allen["ID"] == region_ID]["acronym"].iloc[0]
    print(region, region_ID)
    if "/" in region:
        region = region.replace("/", "_")

    ind = ca.get_vx_ind(region_ID)
    region_mask = np.zeros(ca.voxel_nums, dtype="uint16")
    region_mask[ind] = 1
    region_mask = np.swapaxes(region_mask.reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2)

    # Phase color
    ph_list = CT_df[CT_df["id"] == region_ID]["LAG"].tolist()
    ph = 1 - (ph_list[0] / 24)
    ph = ph + 1 / 3 if (ph + 1 / 3) <= 1 else ph + 1 / 3 - 1
    face_color_r = hsv_to_rgb([ph, 1, 1])
    print(face_color_r)

    region_mask = np.repeat(region_mask, y_scale, axis=1)
    if np.sum(region_mask) == 0:
        continue

    for angle in angles:
        if angle == "hor":
            elevs, azims = [90], [85]
        elif angle == "cor":
            elevs, azims = [0], [90]
        else:  # "sag"
            elevs, azims = [0], [0]

        for elev in elevs:
            for azim in azims:
                fig = plt.figure(figsize=(8, 8))
                ax = fig.add_subplot(111, projection="3d")

                # Whole-brain meshes
                for brain_mask, face_color in zip(brain_masks, face_colors):
                    verts, faces, _, _ = measure.marching_cubes(brain_mask, level=0.5)
                    verts_swapped = verts[:, [2, 1, 0]]
                    mesh = Poly3DCollection(verts_swapped[faces], alpha=0.025)
                    mesh.set_facecolor(face_color)
                    ax.add_collection3d(mesh)

                # Region mesh
                verts, faces, _, _ = measure.marching_cubes(region_mask, level=0.5)
                verts_swapped = verts[:, [2, 1, 0]]
                mesh = Poly3DCollection(verts_swapped[faces], alpha=1.0)
                mesh.set_facecolor(face_color_r)
                ax.add_collection3d(mesh)

                ax.view_init(elev=elev, azim=azim)
                ax.set_xlim(xmin, xmax)
                ax.set_ylim(ymin, ymax)
                ax.set_zlim(zmin, zmax)
                ax.set_xlabel("X")
                ax.set_ylabel("Y")
                ax.set_zlabel("Z")
                ax.set_zlim(ax.get_zlim()[::-1])
                ax.axis("off")

                png_path = os.path.join(fig_dir, f"{region}_{angle}.png")
                fig.savefig(png_path)
                plt.show()
                plt.close()

                # Post-process saved image
                with Image.open(png_path) as img1:
                    if angle == "hor":
                        img1 = img1.rotate(3.0, expand=True, fillcolor=(255, 255, 255))
                        left, top, right, bottom = 280, 220, 590, 640
                    elif angle == "sag":
                        left, top, right, bottom = 215, 270, 645, 565
                    elif angle == "cor":
                        left, top, right, bottom = 230, 247, 603, 603
                        
                    img1 = img1.crop((left, top, right, bottom))
                    fig = plt.figure(figsize=(8, 8))
                    ax = fig.add_subplot()
                    ax.imshow(img1)
                    ax.axis("off")

                    tri_path = os.path.join(fig_dir, f"{region}_{angle}_tri.png")
                    img1.save(tri_path)

In [ ]:
# Generate isometric slices (vx20) of vb regions with borders

bc_dir = os.path.join(savedir, "vx_new")
os.makedirs(bc_dir, exist_ok=True)

region_IDs = [830]

op1 = "fdr"
vb_r = 8
mo = 1
vx = 20
r = 0

angles = ["hor", "cor", "sag"]
sca = 0

sl_num = 7
md = 0
mdc = 0
mdh = 0

zoff = 25 * 20 / vx
yoff = 10 * 20 / vx
xoff = -10 * 20 / vx
offc = 0

sets = [["count", "vb"]]

for typev, unit in sets:
    # Select calc_dir by (typev, unit)
    if unit == "region":
        calc_dir = "whole_region_a" if typev == "count" else "whole_region"
    else:
        calc_dir = "whole_vb_a_new" if typev == "count" else "whole_vb"
            
    for n, region_ID in enumerate(region_IDs):
        op_r = "center" if region_ID in medial_regions else "hemi_right"

        # Per-region manual shifts
        if region_ID in (382, 726, 830):
            s1, s11 = -10, 0
            s2, s22 = 0, 0
            s3, s33 = 0, 0
            sx, sy, sz = 0, 0, 0
        elif region_ID = =1079:  # MGv
            s1, s11 = -10, 0
            s2, s22 = -10, 0
            s3, s33 = 0, 0
            sx, sy, sz = 0, 0, 0
        else:            
            s1, s11 = 0, 0
            s2, s22 = 0, 0
            s3, s33 = 0, 0
            sx, sy, sz = 0, 0, 0

        # Region name
        region = ca.df_allen[ca.df_allen["ID"] == region_ID]["acronym"].iloc[0]
        print(region)
        print(region_ID)
        if "/" in region:
                    region = region.replace("/", "_")

        # Output path (sag view)
        out_dir = os.path.join(bc_dir, calc_dir, f"{vx}um", region, f"vb{vb_r}_mo{mo}")
        output_path = os.path.join(out_dir, f"vb_{op1}_border_sag_{op_r}_iso_tri2_w_ns_nf.png")

        if os.path.exists(output_path):
            continue

        try:
            # Collect IDs (region + children if applicable)
            if not ca.smallID_q(region_ID):
                ID_li = [region_ID]
                child_IDs, child_regions, middle_IDs, middle_regions = ca.get_child_IDs2(region_ID)
                for m_ID in child_IDs + middle_IDs:
                    ID_li.append(m_ID)
            else:
                ID_li = [region_ID]

            # Region mask
            mask0 = np.isin(np.swapaxes((ca.voxel_ID_order_all).reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2), ID_li)
            mask_size = np.sum(mask0)
            print("mask_size", mask_size)

            # Choose padding r (and edge thickness tag) by size
            if mask_size >= 1800000:
                r = 50
                edge_thick ="large"
            elif mask_size  < 1800000 and mask_size >= 18000:
                r = 20
                edge_thick ="small"
            else:
                r = 15
                edge_thick ="small"

            # Source intensity volume (vb / region)
            if unit == "vb":
                img_vx = tifffile.imread(os.path.join(savedir, calc_dir, f"{vx}um", "whole", f"vb{vb_r}_mo{mo}", f"vb_{op1}_img.tif"))
            elif unit == "region":
                img_vx = tifffile.imread(os.path.join(savedir, calc_dir, f"{vx}um", "whole", f"region_{op1}_img.tif"))

            # Edge overlay (thick for large ROIs)
            if region_ID in (726, 382):
                print("large")
                r_edge = np.zeros((ca.z_num, ca.y_num, ca.x_num, 4), dtype="uint8")
                
                edge_mask = tifffile.imread(os.path.join(rdir, f"{vx}um", "edge_mask.tif"))
                edge_list = make_edge(region_ID, mask0)

                edge_mask2, dilated_coords = thicken_specific_lines(edge_mask, edge_list, radius=2.0)

                edge_vx_ind = np.where(edge_mask2 == 1)
                r_edge[edge_vx_ind] = (255, 255, 255, 60)  # other regions border
                r_edge[dilated_coords] = (255, 255, 255, 255)  # border of ROI (thickened)
                r_edge[edge_mask2 == 0] = (0, 0, 0, 0)
                
            else:
                print("smalll")
                r_edge = np.zeros((ca.z_num, ca.y_num, ca.x_num, 4), dtype="uint8")
                
                edge_mask = tifffile.imread(os.path.join(rdir, f"{vx}um", "edge_mask.tif"))
                edge_vx_ind = np.where(edge_mask == 1)
                r_edge[edge_vx_ind] = (255, 255, 255, 60)  # other regions border

                edge_list = make_edge(region_ID, mask0)
                for z, y, x, _ in edge_list:
                    r_edge[z, y, x] = (255, 255, 255, 255)  # border of ROI
                r_edge[edge_mask == 0] = (0, 0, 0, 0)

            # Reference image
            r_img = tifffile.imread(os.path.join(savedir, "1st", "CT0_01", "cfos", f"ANTsR{vx}", "after_ants.tif"))

            # Aspect ratios per view (for later normalization)
            hw_r = np.zeros(3)
            for l, angle in enumerate(angles):
                if op_r == "hemi_right":
                    xmin_l, xmax_l, xmin_r, xmax_r, ymin, ymax, zmin, zmax = mask_range_hemi(mask0)
                else:
                    xmin, xmax, ymin, ymax, zmin, zmax = mask_range(mask0)

                if angle == "hor":
                    if op_r == "hemi_right":
                        xmin, xmax = xmin_l, xmax_l
                    zmin = zmin - s1
                    zmax = zmax - s11
                    hw_r[0] = (ymax - ymin) / (xmax - xmin)

                elif angle == "cor":
                    if op_r == "hemi_right":
                        xmin, xmax = xmin_l, xmax_l
                    ymin = ymin - s2
                    ymax = ymax - s22
                    hw_r[1] = (zmax - zmin) / (xmax - xmin)

                elif angle == "sag":
                    if op_r == "hemi_right":
                        xmin = int(xmin_l - s3)
                        xmax = int(xmax_l - s33)
                    else:
                        xmin = int(xmin - s3)
                        xmax = int(xmax - s33)
                    ymin = ymin - sca
                    ymax = ymax + sca
                    hw_r[2] = (zmax - zmin) / (ymax - ymin)

            max_r = np.max(hw_r)
            print("max_r", max_r)
            maxin = np.argmax(hw_r)
            print("maxin", maxin)

            if region_ID == 302:  # SCs
                max_r = 3.61 / 4.25
            else:
                max_r = np.max(hw_r)

            for l, angle in enumerate(angles):
                # Per-angle output (suffix _s)
                out_dir = os.path.join(bc_dir, calc_dir, f"{vx}um", region, f"vb{vb_r}_mo{mo}")
                output_path = os.path.join(out_dir, f"vb_{op1}_border_{angle}_{op_r}_iso_tri2_w_ns_nf_s.png")

                # Bounding box per angle
                if op_r == "hemi_right":
                    xmin_l, xmax_l, xmin_r, xmax_r, ymin, ymax, zmin, zmax = mask_range_hemi(mask0)
                else:
                    xmin, xmax, ymin, ymax, zmin, zmax = mask_range(mask0)

                if angle == "hor":
                    if op_r == "hemi_right":
                        xmin, xmax = xmin_l, xmax_l
                    zmin = zmin - s1
                    zmax = zmax - s11

                    h = max_r * (xmax - xmin)
                    h_pre = ymax - ymin
                    dh = h - h_pre
                    ymin = ymin - dh / 2
                    ymax = ymax + dh / 2
                    if ymin < 0:
                        ymin = ymin + dh / 2
                        ymax = ymax + dh / 2
                    elif ymax > ca.y_num:
                        ymin = ymin - dh / 2
                        ymax = ymax - dh / 2

                elif angle == "cor":
                    if op_r == "hemi_right":
                        xmin, xmax = xmin_l, xmax_l
                    ymin = ymin - s2
                    ymax = ymax - s22

                    h = max_r * (xmax - xmin)
                    h_pre = zmax - zmin
                    dh = h - h_pre
                    zmin = zmin - dh / 2
                    zmax = zmax + dh / 2
                    if zmin < 0:
                        zmin = zmin + dh / 2
                        zmax = zmax + dh / 2
                    elif zmax > ca.z_num:
                        zmin = zmin - dh / 2
                        zmax = zmax - dh / 2

                elif angle == "sag":
                    if op_r == "hemi_right":
                        xmin = int(xmin_l - s3)
                        xmax = int(xmax_l - s33)
                    else:
                        xmin = int(xmin - s3)
                        xmax = int(xmax - s33)
                    ymin = ymin - sca
                    ymax = ymax + sca

                    h = max_r * (ymax - ymin)
                    h_pre = zmax - zmin
                    dh = h - h_pre
                    zmin = zmin - dh / 2
                    zmax = zmax + dh / 2
                    if zmin < 0:
                        zmin = zmin + dh / 2
                        zmax = zmax + dh / 2
                    elif zmax > ca.z_num:
                        zmin = zmin - dh / 2
                        zmax = zmax - dh / 2

                # Clamp bounds
                if xmin < 0:
                    xmin = 0
                if xmax > ca.x_num:
                    xmax = ca.x_num
                if ymin < 0:
                    ymin = 0
                if ymax > ca.y_num:
                    ymax = ca.y_num
                if zmin < 0:
                    zmin = 0
                if zmax > ca.z_num:
                    zmax = ca.z_num

                # Crop volumes
                r_img_c = r_img[int(zmin):int(zmax), int(ymin):int(ymax), int(xmin):int(xmax)]
                img_vx_c = img_vx[int(zmin):int(zmax), int(ymin):int(ymax), int(xmin):int(xmax)]
                r_edge_c = r_edge[int(zmin):int(zmax), int(ymin):int(ymax), int(xmin):int(xmax)]

                # Plot slices
                fig = plt.figure(figsize=(25, 10))

                if angle == "hor":
                    slice_vs = np.linspace(0, zmax - zmin, sl_num)
                elif angle == "cor":
                    slice_vs = np.linspace(0, ymax - ymin, sl_num)
                else:  # "sag"
                    slice_vs = np.linspace(0, xmax - xmin, sl_num)

                for k, sl in enumerate(slice_vs):
                    if k == 0 or k == sl_num - 1:
                        continue

                    if angle == "hor":
                        ax = fig.add_subplot(1, 5, k + 1 - 1)
                        r_img_c2 = r_img_c[int(sl + mdh), :, :]
                        img_vx_c2 = img_vx_c[int(sl + mdh), :, :]
                        r_edge_c2 = r_edge_c[int(sl + mdh), :, :]
                        ax.imshow(r_img_c2)
                        ax.imshow(img_vx_c2)
                        ax.imshow(r_edge_c2)
                        if k == sl_num - 2:
                            width = r_img_c.shape[0]
                            height = r_img_c.shape[1]
                            base = 1 / ca.x_num * width
                            if base < 0.05:
                                var = 0.05
                            elif 0.05 <= base < 0.1:
                                var = 0.1
                            elif 0.1 <= base < 0.2:
                                var = 0.2
                            elif 0.2 <= base < 0.5:
                                var = 0.5
                            elif 0.5 <= base < 1.0:
                                var = 1.0
                            else:
                                var = 1.0
                            print(angle, height)

                    elif angle == "cor":
                        ax = fig.add_subplot(1, 5, k + 1 - 1)
                        r_img_c2 = r_img_c[:, int(sl + mdc), :]
                        img_vx_c2 = img_vx_c[:, int(sl + mdc), :]
                        r_edge_c2 = r_edge_c[:, int(sl + mdc), :]
                        ax.imshow(r_img_c2)
                        ax.imshow(img_vx_c2)
                        ax.imshow(r_edge_c2)
                        if k == sl_num - 2:
                            width = r_img_c.shape[0]
                            height = r_img_c.shape[1]
                            base = 1 / ca.x_num * width
                            if base < 0.05:
                                var = 0.05
                            elif 0.05 <= base < 0.1:
                                var = 0.1
                            elif 0.1 <= base < 0.2:
                                var = 0.2
                            elif 0.2 <= base < 0.5:
                                var = 0.5
                            elif 0.5 <= base < 1.0:
                                var = 1.0
                            else:
                                var = 1.0
                            print(angle, height)

                    elif angle == "sag":
                        reversed_k = (sl_num - 1) - k
                        ax = fig.add_subplot(1, 5, reversed_k + 1 - 1)
                        ax.imshow(r_img_c[:, :, int(sl + md)])
                        ax.imshow(img_vx_c[:, :, int(sl + md)])
                        ax.imshow(r_edge_c[:, :, int(sl + md)])
                        if k == 1:
                            width = r_img_c.shape[0]
                            height = r_img_c.shape[1]
                            base = 1 / ca.y_num * width
                            if base < 0.05:
                                var = 0.05
                            elif 0.05 <= base < 0.1:
                                var = 0.1
                            elif 0.1 <= base < 0.2:
                                var = 0.2
                            elif 0.2 <= base < 0.5:
                                var = 0.5
                            elif 0.5 <= base < 1.0:
                                var = 1.0
                            else:
                                var = 1.0
                            print(angle, height)

                    ax.axis("off")

                plt.tight_layout()
                os.makedirs(os.path.join(savedir, calc_dir, f"{vx}um", region, f"vb{vb_r}_mo{mo}"), exist_ok=True)
                plt.savefig(os.path.join(bc_dir, f"{vx}_{region}_{angle}_bc_hemi.png"))
                plt.show()

        except:
            traceback.print_exc()

In [ ]:
# Align figures by regions (right hemi or center), with/without scalebar

fig_dir_whole = os.path.join(savedir, "whole_3D")
os.makedirs(fig_dir_whole, exist_ok=True)
ex_dir = os.path.join(savedir, "slices_ex")
db_dir = os.path.join(savedir, "slices")

region_IDs = [909]

calc_dir = "whole_vb_a"
op1 = "fdr"
op = "hemi"
op_r = "hemi_right"
vb_r = 8
mo = 1
vx = 20
vx2 = 20

angles = ["hor", "cor", "sag"]
cut_angles = ["sag", "sag", "hor"]
cut_sets = ["zcut", "ycut", "xcut"]

elevzs = [0]
azimzs = [-3]
elevys = [0]
azimys = [2]


for region_ID in region_IDs:
    region = ca.df_allen[ca.df_allen["ID"] == region_ID]["acronym"].iloc[0]
    print(region_ID, region)

    region_pre = region
    if "/" in region:
        region = region.replace("/", "_")

    try:
        base_dir = os.path.join(savedir, f"{calc_dir}", f"{vx}um", f"{region}", f"vb{vb_r}_mo{mo}")

        img0_path = os.path.join(base_dir, f"vb_{op1}_border_hor_{op_r}_iso_tri2_w.png")
        img0 = Image.open(img0_path)

        if 7 / 316 * img0.height < 1:
            fig = plt.figure(figsize=(18, 2))
        else:
            fig = plt.figure(figsize=(18, 7 / 316 * img0.height))

        gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1])

        region_color = tuple(df_sum[df_sum["id"] == region_ID]["rgb_triplet2"].iloc[0])

        for n, angle in enumerate(angles):
            cut_angle = cut_angles[n]
            cut = cut_sets[n]

            ax = fig.add_subplot(gs[n, 0])
            angle_png = os.path.join(base_dir, f"vb_{op1}_border_{angle}_{op_r}_iso_tri2_w.png")
            with Image.open(angle_png) as img:
                ax.imshow(img)
            ax.axis("off")

        input_path = os.path.join(base_dir, f"vb_{op1}_border_{op_r}_iso_all2_w.png")
        output_path = os.path.join(base_dir, f"vb_{op1}_border_{op_r}_iso_all_tri2_w.png")

        plt.tight_layout()
        plt.subplots_adjust(wspace=0.01, hspace=0.01)
        fig.savefig(input_path, dpi=600)
        fig.savefig(os.path.join(ex_dir, f"{region}_vb{vb_r}_mo{mo}_vb_{op1}_border_{op_r}_iso_all2_w.SVG"))
        trim_image(input_path, output_path, 10)
        plt.close()

        print("size", 6 / 316 * img0.height)

        if 6 / 316 * img0.height < 0.3:
            fig = plt.figure(figsize=(17, 3))
        else:
            fig = plt.figure(figsize=(17, 6 / 316 * img0.height))
        gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.15, 0.7])

        for n, angle in enumerate(angles):
            ax = fig.add_subplot(gs[n, 0])
            tri20 = os.path.join(fig_dir_whole, f"{region}_{angle}_tri_20.png")
            with Image.open(tri20) as img:
                ax.imshow(img)
            ax.axis("off")

        ax = fig.add_subplot(gs[0:, 1])
        with Image.open(output_path) as img:
            ax.imshow(img)
        ax.axis("off")

        fig.savefig(os.path.join(ex_dir, f"3D_{region}_{region}_vb_border_{op_r}_slice_iso_all_w.SVG"))
        plt.show()

        # Non-scalebar
        if 7 / 316 * img0.height < 1:
            fig = plt.figure(figsize=(18, 2))
        else:
            fig = plt.figure(figsize=(18, 7 / 316 * img0.height))

        gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1])

        region_color = tuple(df_sum[df_sum["id"] == region_ID]["rgb_triplet2"].iloc[0])
        for n, angle in enumerate(angles):
            ax = fig.add_subplot(gs[n, 0])
            angle_png_ns = os.path.join(base_dir, f"vb_{op1}_border_{angle}_{op_r}_iso_tri2_w_ns.png")
            with Image.open(angle_png_ns) as img:
                ax.imshow(img)
            ax.axis("off")

        input_path = os.path.join(base_dir, f"vb_{op1}_border_{op_r}_iso_all2_w_ns.png")
        output_path = os.path.join(base_dir, f"vb_{op1}_border_{op_r}_iso_all_tri2_w_ns.png")

        plt.tight_layout()
        plt.subplots_adjust(wspace=0.01, hspace=0.01)
        fig.savefig(input_path, dpi=600)
        fig.savefig(os.path.join(ex_dir, f"{region}_vb{vb_r}_mo{mo}_vb_{op1}_border_{op_r}_iso_all2_w_ns.SVG"))
        trim_image(input_path, output_path, 10)
        plt.show()
        plt.close()

        print("size", 6 / 316 * img0.height)

        if 6 / 316 * img0.height < 0.3:
            fig = plt.figure(figsize=(17, 3))
        else:
            fig = plt.figure(figsize=(17, 6 / 316 * img0.height))
        gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.15, 0.7])

        for n, angle in enumerate(angles):
            ax = fig.add_subplot(gs[n, 0])
            tri20 = os.path.join(fig_dir_whole, f"{region}_{angle}_tri_20.png")
            with Image.open(tri20) as img:
                ax.imshow(img)
            ax.axis("off")

        ax = fig.add_subplot(gs[0:, 1])
        with Image.open(output_path) as img:
            ax.imshow(img)
        ax.axis("off")

        fig.savefig(os.path.join(ex_dir, f"3D_{region}_{region}_vb_border_{op_r}_slice_iso_all_w_ns.SVG"))

        plt.show()
        plt.close()

        trim_image_pil(input_path, output_path)

    except:
        traceback.print_exc()

In [ ]:
# Slice vb regions with borders (wide SCH view, isometric projection)

calc_dir = "whole_vb_a"
op1 = "fdr"
op = "hemi"
vb_r = 8
mo = 1
vx = 20
r = 100

angles = ["hor", "cor", "sag"]

sl_num = 7
md = 0
mdc = 0
mdh = 0

zoff = 25 * 20 / vx
yoff = 10 * 20 / vx
xoff = -10 * 20 / vx
xcoff = 5 * 20 / vx
offc = 5

ID_li = [286]  # SCH

s1 = s11 = s2 = s22 = s3 = s33 = 0

for n, rID in enumerate(ID_li):
    region = ca.df_allen.loc[ca.df_allen["ID"] == rID, "acronym"].iloc[0]
    print(region, rID)

    region = region.replace("/", "_") if "/" in region else region

    if not ca.smallID_q(rID):
        ID_li = [rID]
        child_IDs, _, middle_IDs, _ = ca.get_child_IDs2(rID)
        ID_li += [m_ID for m_ID in (child_IDs + middle_IDs) if m_ID in atlas_ID_li]
    else:
        ID_li = [rID]

    mask0 = np.isin(np.swapaxes(ca.voxel_ID_order_all.reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2), ID_li)

    img_vx = tifffile.imread(f"{savedir}/{calc_dir}/{vx}um/whole/vb{vb_r}_mo{mo}/vb_{op1}_img.tif")

    # Edge mask
    r_edge = np.zeros((ca.z_num, ca.y_num, ca.x_num, 4), dtype="uint8")
    edge_mask = tifffile.imread(f"{rdir}{vx}um/edge_mask.tif")
    r_edge[np.where(edge_mask == 1)] = (255, 255, 255, 100)
    for z, y, x, _ in make_edge(rID, mask0):
        r_edge[z, y, x] = (255, 255, 255, 255)
    tifffile.imwrite(f"{savedir}region_edge/edge_{region}_{vx}um_{r}_vb.tif", r_edge)

    r_img = tifffile.imread(f"{savedir}/1st/CT0_01/cfos/ANTsR{vx}/after_ants.tif")

    # Slice visualization
    for angle in angles:
        if angle == "hor":
            xmin, xmax = int(7363.42 / vx), int(9863.196 / vx)
            ymin, ymax = int(9596.6045 / vx), int(12207.079 / vx)
            zmin, zmax = 447, 478
        elif angle == "cor":
            xmin, xmax = int(7363.42 / vx), int(9863.196 / vx)
            ymin, ymax = 530, 571
            zmin, zmax = int(8519.214 / vx), int(10133.864 / vx)
        else:  # "sag"
            xmin, xmax = 403, 428
            ymin, ymax = int(9596.6045 / vx), int(12207.079 / vx)
            zmin, zmax = int(8519.214 / vx), int(10133.864 / vx)

        slice_vs = np.linspace(0, (zmax - zmin if angle == "hor" else ymax - ymin if angle == "cor" else xmax - xmin), sl_num)
        r_img_c = r_img[zmin:zmax, ymin:ymax, xmin:xmax]
        img_vx_c = img_vx[zmin:zmax, ymin:ymax, xmin:xmax]
        r_edge_c = r_edge[zmin:zmax, ymin:ymax, xmin:xmax]

        fig = plt.figure(figsize=(25, 10))
        for k, sl in enumerate(slice_vs):
            if k in [0, sl_num - 1]:
                continue
            ax = fig.add_subplot(1, 5, k)
            if angle == "hor":
                ax.imshow(r_img_c[int(sl + mdh)], cmap=None)
                ax.imshow(img_vx_c[int(sl + mdh)], cmap=None)
                ax.imshow(r_edge_c[int(sl + mdh)])
            elif angle == "cor":
                ax.imshow(r_img_c[:, int(sl + mdc)], cmap=None)
                ax.imshow(img_vx_c[:, int(sl + mdc)], cmap=None)
                ax.imshow(r_edge_c[:, int(sl + mdc)])
            else:  # "sag"
                ax.imshow(r_img_c[:, :, int(sl + md)], cmap=None)
                ax.imshow(img_vx_c[:, :, int(sl + md)], cmap=None)
                ax.imshow(r_edge_c[:, :, int(sl + md)])

            ax.axis("off")

        plt.tight_layout()
        os.makedirs(f"{savedir}/{calc_dir}/{vx}um/{region}/vb{vb_r}_mo{mo}/", exist_ok=True)
        out_base = f"{savedir}/{calc_dir}/{vx}um/{region}/vb{vb_r}_mo{mo}/vb_{op1}_border_{angle}_{op}"
        plt.savefig(f"{out_base}_iso2.png")
        plt.show()
        trim_image(f"{out_base}_iso2.png", f"{out_base}_iso_tri2.png")

In [ ]:
# Colormap optimized for color vision deficiency

color_points_mod = [
    (0.00, (1.00, 0.45, 0.15)),  # orange
    (0.25, (1.00, 1.00, 0.40)),  # yellowish
    (0.50, (0.60, 0.90, 0.90)),  # light cyan
    (0.75, (0.15, 0.75, 1.00)),  # cyan-blue
    (1.00, (0.20, 0.05, 0.85))   # blue-violet
]

cmap_icefire = LinearSegmentedColormap.from_list(
    "erdc_icefire_darkred",
    color_points_mod,
    N=256
)

fig, ax = plt.subplots(figsize=(8, 1.0))
grad = np.linspace(0, 1, 1200)[None, :]
ax.imshow(grad, aspect='auto', cmap=cmap_icefire, extent=[0, 24, 0, 1])
ax.set_yticks([])
ax.set_xlim(0, 24)
ax.set_xlabel("CT (h)")
ax.set_xticks([0, 6, 12, 18, 24])
plt.tight_layout()